In [9]:
# Import the packages
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
# Create dataframe
df = pd.read_csv('course_lead_scoring.csv')
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


##### Data preperation

In [3]:
# Check if the missing values are presented in the features
# print(df.isna().sum())

# If there are missing values:
# For categorical features, replace them with 'NA'
cat = df.select_dtypes(include='object').columns
df[cat] = df[cat].fillna('NA')

# For numerical features, replace with with 0.0
cat = df.select_dtypes(include='number').columns
df[cat] = df[cat].fillna(0.0)

print(df.isna().sum())

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


##### Question 1: What is the most frequent observation (mode) for the column industry?

In [4]:
# The answer is retail
df['industry'].mode()

0    retail
Name: industry, dtype: object

##### Question 2: what are the two features that have the biggest correlation?

In [5]:
# We can only use the numerical features
# The answer is annual_income and interaction_count
data = df.copy()
data = data.drop('converted', axis=1)

# data.describe() # The difference with the notebook is that he doesn't replace the nan-values

corr_matrix = data.select_dtypes("number").corr()

corr_matrix.unstack().sort_values(ascending=False)

number_of_courses_viewed  number_of_courses_viewed    1.000000
annual_income             annual_income               1.000000
interaction_count         interaction_count           1.000000
lead_score                lead_score                  1.000000
annual_income             interaction_count           0.027036
interaction_count         annual_income               0.027036
annual_income             lead_score                  0.015610
lead_score                annual_income               0.015610
interaction_count         lead_score                  0.009888
lead_score                interaction_count           0.009888
number_of_courses_viewed  annual_income               0.009770
annual_income             number_of_courses_viewed    0.009770
number_of_courses_viewed  lead_score                 -0.004879
lead_score                number_of_courses_viewed   -0.004879
number_of_courses_viewed  interaction_count          -0.023565
interaction_count         number_of_courses_viewed   -0

##### Split the data

In [6]:
# Features and targets
X = df.drop(columns='converted')
y = df['converted']

# First 60% train and 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=42
)

# After that 20% validation and 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

print(len(X_train), len(X_val), len(X_test))

877 292 293


##### Question 3: calculate the mutual information score between the y and x

In [7]:
# The mutual information score is a measure to see the relationship between two variables
# Not a correlation but a non-linear relationship for example
categorical = X_train.select_dtypes(include='object')
mi = categorical.apply(lambda col: mutual_info_score(col, y_train))

# Afronden en sorteren
mi = mi.round(2).sort_values(ascending=False)

print(mi)

# The answer is lead_source

lead_source          0.03
industry             0.02
employment_status    0.02
location             0.00
dtype: float64


##### Question 4: train a logistic regression. What accuracy did you get?

In [10]:
# Create a dictionary for every row
train_dicts = X_train.to_dict(orient='records')
val_dicts   = X_val.to_dict(orient='records')

# One-hot encoding: fit on train, transform on val
dv = DictVectorizer(sparse=False)
X_train_encoded = dv.fit_transform(train_dicts)
X_val_encoded   = dv.transform(val_dicts)


In [13]:
# Train the logistic regression
model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)
model.fit(X_train_encoded, y_train)

# The outcome error is with this max_iter

c:\Users\WilliamvanderAa\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [20]:
# Predict on validation and get the accurary
y_pred = model.predict(X_val_encoded)
original_score = accuracy_score(y_val, y_pred)
accuracy = round(original_score, 2)

print(accuracy)

# The answer is 0.86/0.84

0.86


##### Question 5: find the least userful feature using the feature elimination technique

In [21]:
# Get the features to check which one is the least useful
features = X_train.columns.to_list()
cols_to_check = [
    'industry',
    'employment_status',
    'lead_score'
]

In [ ]:
# Train the model without the eliminates
results = []
for feature in cols_to_check:
    subset = features.copy() # Get a copy from the original variable and the delete every iteration one of the three cols_to_check
    subset.remove(feature)

    train_ft = X_train[subset].to_dict(orient='records')
    val_ft   = X_val[subset].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    Xf_train_encoded = dv.fit_transform(train_ft)
    Xf_val_encoded   = dv.transform(val_ft)

    model_ft = LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42
    )
    model_ft.fit(Xf_train_encoded, y_train)

    yf_pred = model_ft.predict(Xf_val_encoded)
    score = accuracy_score(y_val, yf_pred)

    results.append({
        'eliminated_feature': feature,
        'accuracy': score,
        'difference': original_score - score
    })

scores_df  = pd.DataFrame(results)

# Check the lowest one, the answer is industry
scores_df[scores_df.index == scores_df.difference.idxmin()]

c:\Users\WilliamvanderAa\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\WilliamvanderAa\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preproces

,eliminated_feature,accuracy,difference
0,industry,0.859589,0.0


##### Question 6: which of these C leads to the best accuracy on the validation set?

In [29]:
# Check which C is the best one to use
C_list = [0.01, 0.1, 1, 10, 100]

for c in C_list:
    # Train the logistic regression
    model = LogisticRegression(
        C=c,
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train_encoded, y_train)

    # Predict on validation and get the accurary
    y_pred = model.predict(X_val_encoded)
    original_score = accuracy_score(y_val, y_pred)
    accuracy = round(original_score, 4)

    print(f"Number of c: {c} with an accuracy of {accuracy}")

# The answer is 0.01 or 1

Number of c: 0.01 with an accuracy of 0.8596
Number of c: 0.1 with an accuracy of 0.8562


c:\Users\WilliamvanderAa\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Number of c: 1 with an accuracy of 0.8596
Number of c: 10 with an accuracy of 0.8527
Number of c: 100 with an accuracy of 0.8527


c:\Users\WilliamvanderAa\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
